<a href="https://www.kaggle.com/code/mrafraim/dl-day-40-cnn-multi-label-classification?scriptVersionId=302828470" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 40: CNN Multi-Label Classification
**BCEWithLogits, Threshold Tuning, Precision–Recall Tradeoffs**

Welcome to Day 40!

You’ll Learn Today:

1. Multi-class vs Multi-label (critical distinction)
2. Why Softmax fails in multi-label problems
3. BCEWithLogitsLoss (correct loss)
4. Threshold tuning
5. Precision–Recall tradeoffs
6. Proper evaluation strategy


If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# Multi-Class vs Multi-Label Classification

Understanding the difference is critical because it determines the loss function and evaluation method.

## Multi-Class Classification

Each sample belongs to **exactly one class**.

Classes are **mutually exclusive**.

Example:

- Dog OR Cat OR Car

Not possible to have multiple labels simultaneously.

Typical loss function:

- CrossEntropyLoss

## Multi-Label Classification

Each sample can belong to multiple classes simultaneously.

Classes are independent, not mutually exclusive.

Example:

An image may contain:

✔ Dog  
✔ Person  
✔ Bicycle  

The model must predict **all labels that apply**.

# Why Softmax Fails in Multi-Label Problems

Softmax enforces:

$$
\sum_i P(y=i) = 1
$$

Meaning:

Increasing the probability of one class automatically reduces others.

This assumption only works when classes are **mutually exclusive**.

In multi-label problems:

- Classes are independent.

Example:

Image contains **dog and person** simultaneously.

Softmax cannot represent this.

Therefore:

> Softmax should NOT be used.

# BCEWithLogitsLoss

**BCEWithLogitsLoss = Binary Cross Entropy With Logits Loss**

It is a loss function used for multi-label classification.

## Multi-Label Classification Overview

In multi-label problems:

A sample can belong to **multiple classes at the same time**.

Example classes:

$$
[Dog, Cat, Person, Car]
$$

Image may contain:

$$
Dog + Person
$$

So label becomes:

$$
[1, 0, 1, 0]
$$

Unlike multi-class classification:
- Multi-class → only one class
- Multi-label → multiple classes possible

Because of this, we treat each class as an independent binary classification task.


## Binary Cross Entropy (BCE)

Binary Cross-Entropy measures the difference between:

- true label $y$
- predicted probability $p$

Loss formula:

$$
L = -[y\log(p) + (1-y)\log(1-p)]
$$

Where:


- $y$ = true label (0 or 1)
- $p$ = predicted probability

## What are Logits?


Logits are the raw outputs of a neural network before applying sigmoid or softmax.

Example logits:

$$
[2.1, -0.8, 1.7, -1.2]
$$

They are not probabilities.

Logits can be:

$$
(-\infty, +\infty)
$$

To convert logits into probabilities, we use sigmoid.

## Sigmoid Function

Sigmoid converts logits into probability.

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Output range:

$$
0 \le p \le 1
$$

Example:

| Logit | Probability |
|------|------|
| 2.0 | 0.88 |
| 0 | 0.50 |
| -2.0 | 0.12 |

Large positive logit → high probability.

Large negative logit → low probability.

## Why BCEWithLogitsLoss?

Instead of computing:

$$
Sigmoid → BCE
$$

PyTorch combines both into one function:

`BCEWithLogitsLoss`

So internally it does:

$$
\text{Sigmoid} + \text{Binary Cross Entropy}
$$

Advantages:

✔ Prevents numerical overflow  
✔ Stable gradients  
✔ Faster computation  

PyTorch implementation:

```python
criterion = torch.nn.BCEWithLogitsLoss()
```

## Input Format

Model output shape:

(batch_size, num_classes)

Labels must be multi-hot vectors.

Example:

$$
[1, 0, 1, 0]
$$

Meaning:

| Class  | Label |
| ------ | ----- |
| Dog    | 1     |
| Cat    | 0     |
| Person | 1     |
| Car    | 0     |

## Manual Example

Classes:

$$
[Dog, Cat, Person, Car]
$$

**Ground Truth**
  
$$
y = [1, 0, 1, 0]
$$

**Model Logits**

$$
z = [2.1, -0.8, 1.7, -1.2]
$$

#### Step 1: Apply Sigmoid

Compute:

$$
p = \sigma(z)
$$

Results:

| Class  | Logit | Probability |
| ------ | ----- | ----------- |
| Dog    | 2.1   | 0.89        |
| Cat    | -0.8  | 0.31        |
| Person | 1.7   | 0.85        |
| Car    | -1.2  | 0.23        |

So:

$$
p = [0.89, 0.31, 0.85, 0.23]
$$

#### Step 2: Compute BCE Loss for Each Class

Formula:

$$
L = -[y\log(p) + (1-y)\log(1-p)]
$$


**Dog**

$$
y=1,\ p=0.89
$$

$$
L = -\log(0.89) = 0.116
$$


**Cat**

$$
y=0,\ p=0.31
$$

$$
L = -\log(1-0.31)
$$

$$
= -\log(0.69) = 0.37
$$


**Person**

$$
y=1,\ p=0.85
$$

$$
L = -\log(0.85) = 0.16
$$


**Car**

$$
y=0,\ p=0.23
$$

$$
L = -\log(1-0.23)
$$

$$
= -\log(0.77) = 0.26
$$


#### Step 3: Total Loss

Average loss across classes:

$$
L = \frac{0.116 + 0.37 + 0.16 + 0.26}{4}
$$

$$
L \approx 0.226
$$


#### Final Interpretation

Predicted probabilities:

$$
[0.89, 0.31, 0.85, 0.23]
$$

Interpretation:

| Class  | Prediction |
| ------ | ---------- |
| Dog    | Present    |
| Cat    | Absent     |
| Person | Present    |
| Car    | Absent     |

The model correctly detected:

✔ Dog
✔ Person

and rejected:

✔ Cat
✔ Car

# Threshold Tuning

In multi-label classification, the model outputs probabilities after applying the sigmoid function.

$$
p = \sigma(z)
$$

Where:

- $z$ = model logit (raw output)
- $p$ = probability between $0$ and $1$

But probabilities are not predictions yet.

We must convert probabilities into binary labels.

## Prediction Rule

Binary prediction rule:

$$
\hat{y} =
\begin{cases}
1 & \text{if } p > \text{threshold} \\
0 & \text{otherwise}
\end{cases}
$$

Meaning:

If probability exceeds the threshold → class is predicted as **present**.

## Default Threshold

Most libraries use:

$$
\text{threshold} = 0.5
$$

Example:

| Probability | Prediction |
|-------------|-----------|
| 0.82 | 1 |
| 0.67 | 1 |
| 0.45 | 0 |
| 0.12 | 0 |

But 0.5 is arbitrary.

It assumes:

- balanced classes
- well-calibrated probabilities
- equal cost for false positives and false negatives

These assumptions are often wrong in real datasets.

## Why Threshold Tuning Matters

Consider a rare class.

Example dataset:

- Person appears in 5% of images

The model may output probabilities like:

| Probability |
|-------------|
| 0.35 |
| 0.42 |
| 0.38 |

All are below 0.5, so the model predicts no person every time.

Accuracy may look high, but the model completely misses the minority class.

Lowering the threshold to **0.35** may dramatically improve detection.

## Global vs Per-Class Threshold

### 1. Global Threshold

Same threshold for every class.

Example:

$$
\text{threshold} = 0.5
$$

Simple and easy to implement.

But different classes often have different score distributions.

So one threshold may not work well for all classes.

### 2. Per-Class Threshold

Each class has its own threshold.

Example:

| Class | Threshold |
|------|-----------|
Dog | 0.6 |
Cat | 0.5 |
Person | 0.4 |
Car | 0.7 |

Why?

Some classes are easier to detect, while others require lower thresholds.

Per-class tuning often improves:

- Recall for rare classes
- Overall F1 score
- Balanced performance

## Manual Example.

Classes:

$$ 
[Dog, Cat, Person, Car]
$$

Ground truth:

$$
[1, 0, 1, 0]
$$

Model Probabilities:

$$
[0.62, 0.41, 0.48, 0.30]
$$

**Using default threshold (0.5)**

$$
[1, 0, 0, 0]
$$

Model misses Person.

**Using tuned thresholds**


Dog → 0.6<br>
Cat → 0.5<br>
Person → 0.4<br>
Car → 0.7


Predictions:

$$
[1, 0, 1, 0]
$$

Now the model correctly detects **Person**.

## PyTorch Implementation

```python

# Step 1 — Convert logits to probabilities
probs = torch.sigmoid(outputs)
# Step 2 — Define thresholds
thresholds = [0.6, 0.5, 0.4, 0.7]
# Step 3 — Apply threshold rule
preds = (probs > torch.tensor(thresholds)).float()
```

This produces multi-label predictions like:

$$
[1, 0, 1, 0]
$$


## Summary

Training optimizes loss, not thresholds.

Threshold tuning is a post-training calibration step that converts probabilities into decisions.

In real ML systems, threshold tuning is often required to maximize:

- Precision
- Recall
- F1 score

# Precision–Recall Tradeoff

In classification problems, especially in imbalanced datasets and multi-label tasks,  accuracy becomes misleading.  
Instead, we evaluate models using Precision and Recall, which focus specifically on positive predictions.

These metrics are derived from the confusion matrix:

|                | Predicted Positive | Predicted Negative |
|----------------|-------------------|--------------------|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

## Precision

Precision measures how reliable the model’s positive predictions are.

It answers the question:

> When the model predicts positive, how often is it correct?

Formula:

$$
Precision = \frac{TP}{TP + FP}
$$

Where

- **TP** = correctly predicted positives  
- **FP** = incorrectly predicted positives

High precision means:

- Very few false positives
- Predictions labeled positive are trustworthy

Example:

If a model predicts 100 positive cases and 90 are correct:

$$
Precision = \frac{90}{100} = 0.9
$$

Meaning 90% of positive predictions are correct.

## Recall (Sensitivity / True Positive Rate)

Recall measures the model’s ability to detect all real positive cases.

It answers:

> Out of all actual positives, how many did the model find?

Formula:

$$
Recall = \frac{TP}{TP + FN}
$$

Where

- **TP** = correctly predicted positives  
- **FN** = positives that the model missed

High recall means:

- Very few missed positives
- Model successfully captures most true cases

Example:

If there are 100 real positive cases and the model detects 85:

$$
Recall = \frac{85}{100} = 0.85
$$

Meaning 85% of true positives were detected.


## Precision vs Recall

| Metric | Focus | Penalizes |
|------|------|------|
| Precision | Correctness of positive predictions | False Positives |
| Recall | Coverage of actual positives | False Negatives |

Another way to think:

- Precision → prediction quality
- Recall → detection completeness

## The Precision–Recall Tradeoff

Most models produce probabilities:

$$
P(y=1|x)
$$

To convert probabilities into class predictions, we apply a decision threshold.

Default:

$$
threshold = 0.5
$$

Prediction rule:

$$
\hat{y} =
\begin{cases}
1 & \text{if } P(y=1|x) \ge \text{threshold} \\
0 & \text{otherwise}
\end{cases}
$$

Changing this threshold directly affects precision and recall.

## Threshold Effect

### Lower Threshold

Example:

$$
threshold = 0.2
$$

More samples classified as positive.

Effects:

- ↑ Recall (fewer missed positives)
- ↓ Precision (more false positives)

Reason:

The model becomes less strict when labeling positives.

### Higher Threshold

Example:

$$
threshold = 0.9
$$

Only highly confident predictions are positive.

Effects:

- ↑ Precision
- ↓ Recall

Reason:

The model becomes more conservative.

## Visualization Concept

As threshold changes:

| Threshold | Precision | Recall |
|-----------|----------|--------|
| Low | Low–Medium | High |
| Medium | Balanced | Balanced |
| High | High | Low |

This relationship forms the Precision–Recall curve.

## Real-World Decision Strategy

In real systems, the correct metric depends on cost of errors.

### Medical Diagnosis

Example: cancer screening.

**False Negative (FN)** = patient actually sick but predicted healthy.

This is extremely dangerous.

Goal:

> Maximize Recall

Acceptable tradeoff:

- Some false positives
- But almost no missed disease

### Spam Detection

Example: email filtering.

**False Positive (FP)** = legitimate email goes to spam.

This harms user experience.

Goal:

> Maximize **Precision**

Acceptable tradeoff:

- Some spam may pass through
- But important emails must not be blocked

### Fraud Detection

Banks often balance both:

Goal:

- Catch fraud (**recall**)
- Avoid blocking legitimate customers (**precision**)

Common solution:

- Tune threshold
- Use **F1-score**

## F1 Score (Balanced Metric)

When we need a balance between precision and recall:

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

Properties:

- Harmonic mean of precision and recall
- Penalizes extreme imbalance

Example:

| Precision | Recall | F1 |
|----------|-------|----|
| 0.9 | 0.2 | Low |
| 0.7 | 0.7 | High |

Thus F1 prefers balanced models.

# Threshold Sweeping

In many classification models, especially deep learning models with sigmoid outputs, predictions are probabilities, not final class labels.

Example output:

$$
P(y=1|x) = 0.73
$$

To convert probabilities into class predictions, we apply a decision threshold.

Default threshold:

$$
0.5
$$

Prediction rule:

$$
\hat{y} =
\begin{cases}
1 & \text{if } P(y=1|x) > threshold \\
0 & \text{otherwise}
\end{cases}
$$

However, 0.5 is rarely optimal, particularly for:

- **Imbalanced datasets**
- **Medical detection**
- **Fraud detection**
- **Multi-label classification**

Instead, we use Threshold Sweeping.

## What is Threshold Sweeping?

**Threshold sweeping** means testing multiple thresholds and evaluating model performance at each threshold.

Goal:

> Find the threshold that maximizes validation performance according to a chosen metric.

Common metrics:

- Precision
- Recall
- F1-score
- PR-AUC
- Task-specific metric

## Basic Implementation

```python
import numpy as np
import torch

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    preds = (torch.sigmoid(outputs) > t).float()
```

Explanation:

1. Convert logits → probabilities

```python
probs = torch.sigmoid(outputs)
```

2. Apply threshold

```python
preds = (probs > t).float()
```

3. Compute evaluation metrics


## Complete Example

```python
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import torch

probs = torch.sigmoid(outputs).cpu().numpy()
y_true = labels.cpu().numpy()

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    
    preds = (probs > t).astype(int)

    precision = precision_score(y_true, preds)
    recall = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)

    print(f"Threshold: {t:.1f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
```

Output example:

```
Threshold: 0.1 | Precision: 0.42 | Recall: 0.95 | F1: 0.58
Threshold: 0.3 | Precision: 0.63 | Recall: 0.81 | F1: 0.71
Threshold: 0.5 | Precision: 0.78 | Recall: 0.65 | F1: 0.71
Threshold: 0.7 | Precision: 0.91 | Recall: 0.42 | F1: 0.57
```


## Selecting the Best Threshold

Choose the threshold based on the **objective of the problem**.

Examples:

### Maximize Recall

Used in:

* disease detection
* safety systems

Choose threshold where:

$$
Recall \text{ is highest}
$$


### Maximize Precision

Used in:

* spam filters
* recommendation alerts

Choose threshold where:

$$
Precision \text{ is highest}
$$


### Balanced Performance

Use **F1-score**:

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

Select threshold giving **maximum F1**.

## Visualization (Best Practice)

A better strategy is plotting the **Precision–Recall curve**.

```python
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_true, probs)
```

This helps visualize how precision and recall change with threshold.

## Important Real-World Rule

Always tune threshold using:

**validation set - NOT training set**

Pipeline:

```markdown
        
Train model
     ↓
Predict probabilities on validation data
     ↓
Sweep thresholds
     ↓
Select best threshold
     ↓
Evaluate on test data
```

Reason:

Using training data causes **overfitting of the threshold**.


# Handling Class Imbalance

In real-world datasets, especially multi-label classification, class distributions are rarely balanced.

Some labels appear very frequently, while others occur very rarely.

Example:

| Label | Frequency |
|------|------|
| Person | 80% |
| Car | 60% |
| Dog | 20% |
| Bicycle | 5% |
| Ambulance | 0.3% |

A model trained normally will tend to ignore rare classes, because predicting the majority class already minimizes loss.

This is called the **class imbalance problem**.


## Why Class Imbalance is Dangerous

If a rare label appears only **1% of the time**, the model can achieve:

$$
Accuracy = 99\%
$$

simply by never predicting that label.

But the model would be useless for detecting that class.

Thus we must rebalance the learning signal.

## Strategy: Positive Weighting

For multi-label classification, the most common solution is positive weighting in the loss function.

PyTorch provides this directly in:

`BCEWithLogitsLoss`

The idea:

> Give higher loss penalty when the model misses a rare positive label.

This forces the network to pay more attention to rare classes.


### BCEWithLogitsLoss with Positive Weights

```python
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights)
```

Where:

* `class_weights` is a tensor containing weight per class

Example:

```python
class_weights = torch.tensor([1.0, 1.0, 5.0, 8.0])
```

Interpretation:

| Class   | Weight     |
| ------- | ---------- |
| Class 1 | Normal     |
| Class 2 | Normal     |
| Class 3 | 5× penalty |
| Class 4 | 8× penalty |

Thus:

* Missing **Class 4** is penalized **8 times more** than common classes.


## Mathematical Effect

The weighted BCE loss becomes:

$$
Loss = -[w_p \cdot y \log(\sigma(x)) + (1-y)\log(1-\sigma(x))]
$$

Where:

* $y$ = true label
* $\sigma(x)$ = sigmoid prediction
* $w_p$ = positive weight

Effect:

If a rare positive label is missed, the loss increases significantly.

This pushes the model to increase recall for rare classes.

# Micro vs Macro F1

In multi-class and multi-label classification, evaluating a model with a single metric can be misleading, especially when class distributions are imbalanced.

Two commonly used evaluation strategies are:

- Micro F1
- Macro F1

These differ in how they aggregate performance across classes.


## Micro F1

Micro F1 aggregates TP, FP, and FN across all classes first, then computes the F1 score.

This means every prediction contributes equally, regardless of which class it belongs to.

**Formula:**

First aggregate counts across all classes:

$$
TP_{micro} = \sum TP_i
$$

$$
FP_{micro} = \sum FP_i
$$

$$
FN_{micro} = \sum FN_i
$$

Then compute:

$$
Precision_{micro} = \frac{TP_{micro}}{TP_{micro}+FP_{micro}}
$$

$$
Recall_{micro} = \frac{TP_{micro}}{TP_{micro}+FN_{micro}}
$$

Finally:

$$
F1_{micro} =
2 \times
\frac{Precision_{micro} \times Recall_{micro}}
{Precision_{micro}+Recall_{micro}}
$$


**Interpretation:**

Micro F1 measures overall prediction performance across the entire dataset.

Important characteristic:

> Frequent classes dominate the metric.

If a dataset contains:

| Class | Samples |
|------|------|
| Person | 50,000 |
| Car | 40,000 |
| Dog | 8,000 |
| Ambulance | 50 |

Even if the model performs poorly on Ambulance, Micro F1 may still be high because Person and Car dominate the dataset.

**When It Is Used in Real Systems**

Micro F1 is commonly used when the goal is overall system performance.

Typical applications:

- Large-scale recommendation systems
- Search ranking
- General classification benchmarks

Why?

Because these systems care more about total prediction accuracy than rare-class fairness.


## Macro F1

Macro F1 computes F1-score independently for each class and then averages them.

Each class contributes equally, regardless of how many samples it has.

**Formula:**

First compute per-class F1:

$$
F1_i =
2 \times
\frac{Precision_i \times Recall_i}
{Precision_i + Recall_i}
$$

Then average:

$$
Macro\ F1 =
\frac{1}{N}
\sum_{i=1}^{N} F1_i
$$

where:

- $N$ = number of classes

**Interpretation:**

Macro F1 answers:

> How well does the model perform across all classes equally?

Rare classes receive the same importance as common classes.

**When It Is Used in Real Systems**

Macro F1 is used when every class matters, even if it is rare.

Typical domains:

- **Medical diagnosis**
- **Fraud detection**
- **Defect detection**
- **Safety-critical AI**

Reason:

Failing on a rare class may have serious consequences.


## Example Comparison

Dataset:

| Class | Samples | F1 |
|------|------|------|
| Person | 50000 | 0.95 |
| Car | 40000 | 0.92 |
| Dog | 8000 | 0.88 |
| Ambulance | 50 | 0.10 |

### Micro F1

Dominated by large classes:

$$
Micro\ F1 \approx 0.92
$$

### Macro F1

Equal weighting:

$$
Macro\ F1 =
\frac{0.95 + 0.92 + 0.88 + 0.10}{4}
= 0.71
$$

Macro F1 exposes the failure on the rare class.


# Practical Multi-Label CNN Pipeline

- Final layer → Linear(features, num_classes)
- Loss → BCEWithLogitsLoss
- During inference → Apply sigmoid
- Tune threshold using validation set
- Evaluate using F1 or mAP

# Key Takeaways from Day 40

- Multi-label ≠ multi-class classification
- Softmax is incorrect for multi-label problems
- Use BCEWithLogitsLoss
- Each class behaves like independent binary classifier
- Threshold tuning controls precision–recall balance
- Accuracy is not reliable for evaluation

---
<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
